# 4. Diagnóstico de residuos

Analizamos los residuos `resid = y_real − y_pred` del primer paso del horizonte (h = 1). Aplicamos el test BDS (Broock et al., 1996), cuya hipótesis nula es que los residuos son independientes e idénticamente distribuidos: un p-valor mayor que 0.05 no rechaza esa hipótesis y es la señal deseada. Lo calculamos sobre validación (por fold y estrategia) y sobre el hold-out del modelo final, y lo complementamos con Ljung-Box y la ACF de los residuos.

In [ ]:
# Rutas: en Colab se usa la carpeta de Drive; en el repositorio, la raíz del proyecto.
import os, glob, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

DRIVE = Path("/content/drive/MyDrive/tarea2_series")
if DRIVE.exists():
    BASE, FIGS = DRIVE, DRIVE / "figs"
else:
    BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    FIGS = BASE / "notebooks" / "figs"
RESULTS = BASE / os.environ.get("T2_RESULTS_DIR", "results")
FIGS.mkdir(parents=True, exist_ok=True)

LAGS, HS, ESTRATEGIAS = [7, 14, 21, 28], [7, 14, 21, 28], ["gk", "fc", "kf"]
NOMBRE = {"gk": "Group K-Fold", "fc": "Forward Chaining", "kf": "K-Fold"}
COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]          # orden categorico fijo
COL_EST = dict(zip(ESTRATEGIAS, COL))
COL_LAG = dict(zip(LAGS, COL))
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
                     "axes.prop_cycle": plt.cycler(color=COL), "lines.linewidth": 1.6,
                     "legend.frameon": False, "axes.titlesize": 11, "axes.labelsize": 10})

df = pd.read_csv(BASE / "data" / "btc_1d.csv", parse_dates=["date"])
y = df["close"].to_numpy(float)
DATES = df["date"].to_numpy()
N_HOLD = max(int(len(y) * 0.10), 56)
N_CV = len(y) - N_HOLD


def guardar(fig, nombre):
    fig.savefig(FIGS / f"{nombre}.png", bbox_inches="tight", dpi=150)

In [ ]:
def cargar_registros():
    R = {}
    for f in sorted(glob.glob(str(RESULTS / "raw" / "*.jsonl"))):
        for line in open(f):
            if line.strip():
                r = json.loads(line); R[r["id"]] = r
    return list(R.values())

REG = cargar_registros()
print(f"{len(REG)} registros en {RESULTS / 'raw'}")
print(pd.Series([r["phase"] for r in REG]).value_counts().to_string())

## 4.1 BDS sobre validación

In [ ]:
DET = {(r["L"], r["H"], r["strat"]): r for r in REG if r["phase"] == "detail"}
filas = []
for (L, H, s), d in DET.items():
    p = [f["bds_p_h1"] for f in d["folds"] if f["bds_p_h1"] is not None]
    filas.append(dict(estrategia=NOMBRE[s], lag=L, H=H, folds_con_bds=len(p),
                      bds_p_medio=np.mean(p) if p else np.nan,
                      folds_p_mayor_005=int(np.sum(np.array(p) > 0.05)) if p else 0))
BDSV = pd.DataFrame(filas).sort_values(["estrategia", "lag", "H"])
BDSV.to_csv(RESULTS / "bds_validacion.csv", index=False)
BDSV[BDSV.estrategia == "Group K-Fold"].pivot(index="lag", columns="H", values="bds_p_medio").round(3)

Forward Chaining y K-Fold validan sobre una ventana por fold: un solo residuo h = 1 por fold no permite aplicar BDS, por eso esas filas aparecen vacías.

## 4.2 BDS y Ljung-Box sobre el hold-out del modelo final

In [ ]:
from statsmodels.tsa.stattools import bds
from statsmodels.stats.diagnostic import acorr_ljungbox
filas, RESID = [], {}
for L in LAGS:
    for H in HS:
        f = RESULTS / "models" / f"own_L{L}_H{H}_preds.npz"
        if not f.exists():
            continue
        z = np.load(f); e = z["real"][:, 0] - z["pred"][:, 0]; RESID[(L, H)] = e
        p_bds = float(np.atleast_1d(bds(e, max_dim=2)[1])[0])
        p_lb = float(acorr_ljungbox(e, lags=[10], return_df=True)["lb_pvalue"].iloc[0])
        filas.append(dict(lag=L, H=H, n=len(e), media_resid=e.mean(), std_resid=e.std(),
                          BDS_pvalue_h1=p_bds, LjungBox10_pvalue=p_lb))
BDSH = pd.DataFrame(filas)
BDSH.to_csv(RESULTS / "bds_holdout.csv", index=False)
BDSH.round(4)

## 4.3 Residuos del hold-out (lag 14, H 14)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
def figura_residuos(L, H):
    e = RESID[(L, H)]; t = DATES[N_CV:N_CV + len(e)]
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
    ax[0].plot(t, e, color=COL[0], lw=1); ax[0].axhline(0, color="0.4", lw=0.8)
    ax[0].set_title(f"Residuo h = 1 en el hold-out (lag {L}, H {H})"); ax[0].set_ylabel("USD")
    ax[1].hist(e, bins=40, color=COL[0], alpha=0.8); ax[1].set_title("Distribución")
    plot_acf(e, lags=30, ax=ax[2], title="ACF del residuo")
    fig.tight_layout(); guardar(fig, f"residuos_holdout_L{L}_H{H}"); plt.show()

if (14, 14) in RESID:
    figura_residuos(14, 14)

In [ ]:
for (L, H) in sorted(RESID):
    if (L, H) != (14, 14):
        figura_residuos(L, H)